# 나만의 물체를 찾는 드론 만들기
### YOLO11n 파인튜닝 → Raspberry Pi AI Camera(IMX500) 탑재

기본 모델은 COCO 80종(사람, 자동차, 병...)만 압니다.
여기서는 **내가 직접 모은 사진**으로 학습시켜서 원하는 물체를 찾게 만듭니다.

**전체 흐름**

```
[Colab]  사진 수집 → 라벨링 → 학습 → 양자화 → packerOut.zip
                                                    ↓ 다운로드
[Pi]     imx500-package → network.rpk → AI 카메라에 업로드 → 추적 비행
```

> ⚠️ 마지막 `.rpk` 포장 단계는 Colab에서 할 수 없습니다.
> 포장 도구(`imx500-tools`)가 ARM 전용이라 **라즈베리파이에서** 돌려야 합니다.
> 이 노트북은 그 직전 단계인 `packerOut.zip` 까지 만들어 줍니다.

**시작 전에:** 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

## 0. GPU 확인

In [ ]:
!nvidia-smi -L
print('위에 Tesla T4 같은 게 안 보이면 런타임 유형을 GPU로 바꾸세요!')

## 1. 설치

`ultralytics` 하나만 깔면 학습이 되고, IMX500 변환에 필요한 도구는
나중에 export 할 때 자동으로 설치됩니다. (5분 정도 걸립니다)

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 2. 데이터셋 준비

아래 **A / B / C 중 하나만** 실행하세요.

| 방법 | 언제 쓰나 |
|---|---|
| **A. 연습용 샘플** | 처음이라 전체 흐름부터 돌려보고 싶을 때 |
| **B. Roboflow** | 웹에서 라벨링했을 때 (권장) |
| **C. ZIP 업로드** | 이미 YOLO 형식으로 정리해둔 폴더가 있을 때 |

### YOLO 데이터셋 형식
```
my_dataset/
  data.yaml          <- 클래스 이름 목록
  train/images/*.jpg
  train/labels/*.txt <- 한 줄에 "클래스번호 x중심 y중심 너비 높이" (0~1 비율)
  valid/images/*.jpg
  valid/labels/*.txt
```

**사진은 몇 장이나?** 클래스당 최소 100장, 200~300장이면 꽤 잘 됩니다.
드론에서 쓸 거라면 **위에서 내려다본 각도**, **멀리 있는 모습** 사진을 꼭 넣으세요.

### A. 연습용 샘플 데이터 (COCO8 - 이미지 8장짜리 장난감 데이터)

In [ ]:
# 흐름 연습용입니다. 실제로 쓸 만한 모델은 나오지 않습니다.
DATA_YAML = 'coco8.yaml'
print('연습용 데이터 선택됨')

### B. Roboflow 에서 가져오기

1. https://roboflow.com 가입 → New Project → Object Detection
2. 사진 업로드 → 박스로 라벨링 → Generate → **Export → YOLOv11 → show download code**
3. 나오는 코드의 값들을 아래에 붙여넣기

In [ ]:
# !pip install -q roboflow
# from roboflow import Roboflow
#
# rf = Roboflow(api_key='여기에_API_KEY')
# project = rf.workspace('워크스페이스이름').project('프로젝트이름')
# dataset = project.version(1).download('yolov11')
#
# DATA_YAML = dataset.location + '/data.yaml'
# print(DATA_YAML)

### C. ZIP 파일 직접 업로드

왼쪽 폴더 아이콘 → 업로드로 `my_dataset.zip` 을 올린 뒤 실행하세요.

In [ ]:
# !unzip -q /content/my_dataset.zip -d /content/dataset
# DATA_YAML = '/content/dataset/data.yaml'
#
# # data.yaml 안의 경로가 맞는지 확인
# print(open(DATA_YAML).read())

## 3. 학습 설정

**여기 숫자들만 바꿔가며 실험해 보세요.**

In [ ]:
# 학습을 몇 바퀴 돌릴지. 적으면 덜 배우고, 너무 많으면 외워버립니다(과적합).
EPOCHS = 60

# 입력 이미지 크기. 작을수록 빠르고, 클수록 작은 물체를 잘 찾습니다.
# IMX500 에서는 320 또는 640 을 쓰세요. 드론용은 320 을 추천합니다(빠름).
IMGSZ = 320

# 한 번에 몇 장씩 볼지. GPU 메모리가 부족하다는 에러가 나면 줄이세요.
BATCH = 32

# 베이스 모델. yolo11n 이 가장 작고 빠릅니다 (IMX500 은 n 크기만 지원)
BASE_MODEL = 'yolo11n.pt'

# 결과물 이름
RUN_NAME = 'my_tracker'

## 4. 학습

GPU 기준 60 에폭에 10~30분 정도 걸립니다. 커피 한 잔 ☕

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name=RUN_NAME,
    patience=20,        # 20 에폭 동안 나아지지 않으면 조기 종료
    plots=True,
)

BEST = f'runs/detect/{RUN_NAME}/weights/best.pt'
print('\n학습 완료:', BEST)

## 5. 잘 배웠는지 확인

**mAP50 이 0.7 이상**이면 실전에 써볼 만합니다.
낮다면 → 사진을 더 모으거나, 라벨링이 틀린 곳이 없는지 확인하세요.

In [ ]:
from IPython.display import Image, display

metrics = YOLO(BEST).val(data=DATA_YAML, imgsz=IMGSZ)
print(f'\nmAP50    = {metrics.box.map50:.3f}   <- 0.7 이상이면 good')
print(f'mAP50-95 = {metrics.box.map:.3f}')

display(Image(f'runs/detect/{RUN_NAME}/results.png', width=900))
display(Image(f'runs/detect/{RUN_NAME}/confusion_matrix_normalized.png', width=600))

In [ ]:
# 검증 이미지에 실제로 그려보기
import glob

for p in sorted(glob.glob(f'runs/detect/{RUN_NAME}/val_batch*_pred.jpg'))[:2]:
    display(Image(p, width=900))

## 6. IMX500 용으로 변환 (양자화)

AI 카메라 칩은 소수점 계산을 못 합니다. 그래서 모델을 **정수(8비트)로 압축**해야 합니다.
이 과정을 양자화(quantization)라고 하고, `data=` 로 준 사진들을 기준으로 자동 보정합니다.

필요한 도구가 자동 설치되므로 **10~20분** 걸립니다. 중간에 끊지 마세요.

In [ ]:
model = YOLO(BEST)
export_dir = model.export(format='imx', data=DATA_YAML, imgsz=IMGSZ)
print('\n변환 결과 폴더:', export_dir)

!ls -la {export_dir}

## 7. 다운로드

`packerOut.zip` 과 `labels.txt` 두 개가 필요합니다. 한 파일로 묶어서 받습니다.

In [ ]:
import os, shutil
from google.colab import files

OUT = '/content/imx500_out'
os.makedirs(OUT, exist_ok=True)

shutil.copy(os.path.join(export_dir, 'packerOut.zip'), OUT)
shutil.copy(os.path.join(export_dir, 'labels.txt'), OUT)
shutil.copy(BEST, os.path.join(OUT, 'best.pt'))   # 나중에 재학습용 백업

print('클래스 목록 (config.yaml 의 target_class 에 이 이름 중 하나를 쓰세요):')
print(open(os.path.join(OUT, 'labels.txt')).read())

shutil.make_archive(f'/content/{RUN_NAME}_imx500', 'zip', OUT)
files.download(f'/content/{RUN_NAME}_imx500.zip')

## 8. 라즈베리파이에서 마무리

받은 zip 을 라즈베리파이로 옮긴 뒤 (USB, `scp`, 또는 그냥 웹 다운로드):

```bash
# 1) 압축 풀기
mkdir -p ~/models && cd ~/models
unzip ~/my_tracker_imx500.zip

# 2) .rpk 로 포장  (imx500-tools 는 install.sh 에서 이미 설치됨)
imx500-package -i packerOut.zip -o .
#   -> ~/models/network.rpk 생성

# 3) 라벨 파일 옮기기
cp labels.txt ~/ai-tracking-drone/assets/my_labels.txt
```

그리고 `config.yaml` 을 이렇게 고칩니다:

```yaml
camera:
  model: /home/pi/models/network.rpk
  labels: assets/my_labels.txt

detection:
  target_class: <labels.txt 안의 이름>
  bbox_normalization: true   # YOLO 계열은 반드시 true
  bbox_order: xy             # YOLO 계열은 반드시 xy
  postprocess: ""
```

마지막으로 확인:

```bash
python3 tools/check_camera.py
```

> 처음 실행 시 카메라에 새 펌웨어를 올리느라 1~2분 멈춰 있습니다. 정상입니다.

---
## 잘 안 될 때

| 증상 | 해결 |
|---|---|
| 학습 mAP 가 너무 낮음 | 사진 수 부족. 클래스당 200장 이상, 다양한 각도/거리/조명 |
| 학습은 잘 됐는데 카메라에선 못 찾음 | 양자화 손실. `imgsz=640` 으로 다시 시도 |
| export 에서 에러 | 런타임 재시작 후 6번 셀만 다시 실행 |
| CUDA out of memory | `BATCH` 를 16 또는 8 로 |
| 드론에서만 못 찾음 | 학습 데이터에 **드론 시점(위에서 아래)** 사진이 없는 것. 직접 찍어서 추가 |